
# Building Chat Interfaces with Gradio, Streamlit & Backend Orchestration





## Exercise 1 : Gradio Interface – Multi‑Function Toolkit

Create a simple multi‑function application in **Gradio** using `gr.Interface`.  The user can choose one of three functions via a dropdown:

- **Greet** – takes a name and returns a greeting like `"Hello, [name]!"`.
- **Echo** – takes text and returns `"You said: [text]"`.
- **Square a Number** – takes a number and returns its square.

### Requirements

* Provide a dropdown with the options “Greet”, “Echo” and “Square a Number”.
* Show a text box when a text‑based option is chosen, or a number box when the square option is chosen.
* Dispatch the input to the appropriate backend function based on the option.
* Include example inputs for each function to demonstrate usage.

Below is a sample implementation:


In [ ]:

import gradio as gr

# Define backend functions
def greet(name: str) -> str:
    return f"Hello, {name}!"

def echo(text: str) -> str:
    return f"You said: {text}"

def square_number(num: float) -> float:
    return num ** 2

# Wrapper dispatch function
def multi_tool(choice: str, inp: str) -> str:
    if choice == "Greet":
        return greet(inp)
    elif choice == "Echo":
        return echo(inp)
    elif choice == "Square a Number":
        try:
            num = float(inp)
        except ValueError:
            return "Please enter a valid number"
        return str(square_number(num))
    else:
        return "Unknown option"

with gr.Blocks() as demo:
    gr.Markdown("### Multi‑Function Toolkit")
    choice = gr.Radio([
        "Greet", "Echo", "Square a Number"
    ], label="Choose a function")
    input_box = gr.Textbox(label="Input")
    output_box = gr.Textbox(label="Output", interactive=False)

    def update_input_ui(selected):
        if selected == "Square a Number":
            return gr.update(lines=1, label="Number", placeholder="Enter a number")
        else:
            return gr.update(lines=1, label="Text", placeholder="Enter your text")

    choice.change(update_input_ui, inputs=choice, outputs=input_box)
    submit = gr.Button("Run")

    def run(choice, inp):
        return multi_tool(choice, inp)

    submit.click(run, inputs=[choice, input_box], outputs=output_box)
    gr.Examples([
        ["Greet", "Raphael"],
        ["Echo", "I'm talking!"],
        ["Square a Number", "4"],
    ], inputs=[choice, input_box], outputs=output_box, fn=run)

# To preview locally, run: demo.launch()



## Exercise 2 : Gradio Blocks – Simple Two‑Step Calculator

In this exercise you will build a two‑step calculator with **Gradio Blocks**.  The UI should display two number input fields side by side, allow the user to choose between *Add* and *Multiply* via a radio button, and compute the result when a button is pressed.

### Requirements

* Use a `gr.Row()` container to align the two number inputs horizontally.
* Provide a radio selector with the labels “Add” and “Multiply”.
* Use a button to trigger the calculation.
* Display the result using `gr.Label()` at the bottom.
* Add a short Markdown description at the top.

Below is a sample implementation:


In [ ]:

import gradio as gr

# Simple calculator logic
def calculate(x: float, y: float, operation: str) -> float:
    if operation == "Add":
        return x + y
    else:
        return x * y

with gr.Blocks() as calc:
    gr.Markdown("### Two‑Step Calculator")
    with gr.Row():
        num1 = gr.Number(label="Number 1", value=0)
        num2 = gr.Number(label="Number 2", value=0)
    op = gr.Radio(["Add", "Multiply"], label="Operation", value="Add")
    button = gr.Button("Calculate")
    result = gr.Label(label="Result")

    button.click(calculate, inputs=[num1, num2, op], outputs=result)

# To preview locally, run: calc.launch()



## Exercise 3 : Streamlit Chat – Stateful Memory Bot

Use **Streamlit** to build a simple chat interface that remembers past messages across interactions.  Streamlit’s `st.chat_message()` function draws chat bubbles, and `st.session_state` can store conversation history.

### Requirements

* Display previous messages using `st.chat_message()` and loop through `st.session_state["chat_history"]`.
* Use an input widget (e.g. `st.chat_input`) to collect user messages.
* Append both the user’s message and the bot’s reply to the history list.  For this exercise, the bot can simply echo the user’s message.
* Add a “Clear Chat” button to reset the conversation.

Below is an example script:


In [ ]:

import streamlit as st

st.title("Stateful Chat Bot")

# Initialize chat history
def init_session():
    if "chat_history" not in st.session_state:
        st.session_state["chat_history"] = []

init_session()

# Display previous messages
for role, message in st.session_state["chat_history"]:
    st.chat_message(role).write(message)

# Chat input
user_input = st.chat_input("Type your message…")

if user_input:
    # Append user message
    st.session_state["chat_history"].append(("user", user_input))
    # Simple echo bot reply
    bot_reply = f"You said: {user_input}"
    st.session_state["chat_history"].append(("assistant", bot_reply))
    st.experimental_rerun()

# Clear chat button
if st.button("Clear Chat"):
    st.session_state["chat_history"] = []
    st.experimental_rerun()



## Exercise 4 : Streamlit UI + Logic Combo

Create a mini dashboard in **Streamlit** that displays different content based on user choice.  Use dynamic rendering to either show a code snippet or a JSON object, alongside a random line chart.  Optionally load an image from the internet.

### Requirements

* Display a random line chart using NumPy (`np.random.randn`) and `st.line_chart`.
* Provide a radio button with choices “Show Code” and “Show JSON”.
* If “Show Code” is selected, display a code block using `st.code()`.
* If “Show JSON” is selected, display a JSON object using `st.json()`.
* **Bonus:** fetch an image from a public URL with `st.image()` and display it.

Below is a sample implementation:


In [ ]:

import streamlit as st
import numpy as np

st.title("Mini Dashboard")

# Chart section
st.subheader("Random Line Chart")
chart_data = np.random.randn(20, 3)
st.line_chart(chart_data)

# Choice section
choice = st.radio("Select view", ["Show Code", "Show JSON"])

code_example = "# Example function

def add(a, b):
    return a + b"
json_example = {"name": "Streamlit", "version": "1.x", "features": ["widgets", "charts", "state"]}

if choice == "Show Code":
    st.code(code_example, language="python")
else:
    st.json(json_example)

# Bonus: display an image from a URL
image_url = "https://images.pexels.com/photos/459225/pexels-photo-459225.jpeg"
st.image(image_url, caption="A serene landscape", use_column_width=True)



## Exercise 5 : FastAPI – Smart Responder

Implement a minimal **FastAPI** backend that returns different messages based on keywords in the request.  You will create a `/respond` endpoint that accepts a JSON payload with a `"message"` field and returns a corresponding response.

### Requirements

* Create a POST endpoint `/respond` that accepts a JSON body like `{"message": "What’s 5+5?"}`.
* If the message contains the word *math*, return `{"response": "Using calculator tool..."}`.
* If the message contains the word *date*, return `{"response": "Fetching current date..."}`.
* For all other messages, return `{"response": "Default LLM response."}`.
* Test the API locally using `curl` or a tool like Postman.

Below is a sample FastAPI application:


In [ ]:

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Message(BaseModel):
    message: str

@app.post("/respond")
async def respond(msg: Message):
    text = msg.message.lower()
    if "math" in text:
        return {"response": "Using calculator tool..."}
    elif "date" in text:
        return {"response": "Fetching current date..."}
    else:
        return {"response": "Default LLM response."}

# To run this app, save it as `main.py` and execute:
# uvicorn main:app --reload

# Example curl request:
# curl -X POST -H "Content-Type: application/json" #      -d '{"message": "What's the date today?"}' #      http://127.0.0.1:8000/respond
